# CLINC150 -> bộ dữ liệu intent tiếng Việt (20 intent ngân hàng/thẻ)

**Quy trình:** lọc 20 intent -> tạo danh sách câu -> dịch bằng Gemini -> gộp, kiểm tra, khử trùng lặp -> lưu `subset20_vi.json`.

**Chuẩn bị:**
1. Upload `data_full.json` (từ repo clinc/oos-eval, thư mục `data/`) vào panel Tệp của Colab.
2. Thêm secret `GEMINI_API_KEY` (biểu tượng chìa khóa bên trái, bật *Notebook access*). **Không dán key vào code hay chat.**
3. Chạy lần lượt từng cell. Mặc định `TEST_MODE = True` (chỉ dịch 100 câu đầu để kiểm tra chất lượng).

In [ ]:
!pip -q install google-genai

## 1. Cấu hình

In [ ]:
import json, os, random, re, time, glob
from collections import Counter

DATA_FILE   = "data_full.json"
MODEL       = "gemini-2.5-flash"   # nếu báo lỗi model not found, đổi sang tên model có trong AI Studio
BATCH_SIZE  = 50
TEST_MODE   = True                 # True: chỉ dịch 100 câu đầu. Kiểm tra chất lượng xong thì đổi False
N_TEST_SENT = 100
N_OOS_TEST  = 300                  # số câu OOS test lấy mẫu để dịch (gốc có 1000)
SEED        = 42

KEEP = {
    # ngân hàng (13)
    "account_blocked", "balance", "bill_balance", "bill_due", "freeze_account",
    "interest_rate", "min_payment", "pay_bill", "pin_change", "report_fraud",
    "spending_history", "transactions", "transfer",
    # thẻ (7)
    "card_declined", "credit_limit", "credit_score", "expiration_date",
    "international_fees", "new_card", "report_lost_card",
}
assert len(KEEP) == 20, len(KEEP)

## 2. Đọc dữ liệu gốc và kiểm tra tên intent

In [ ]:
d = json.load(open(DATA_FILE))
print(d.keys())
cnt = Counter(x[1] for x in d["train"])
print(len(cnt), "intent")

missing = KEEP - set(cnt)
assert not missing, f"Sai tên intent: {missing}"
print("OK: 20 intent đều có trong dữ liệu")

## 3. Lọc 20 intent, lấy mẫu OOS test, tạo danh sách câu cần dịch

In [ ]:
random.seed(SEED)

out = {k: [x for x in d[k] if x[1] in KEEP] for k in ["train", "val", "test"]}
for k in ["oos_train", "oos_val", "oos_test"]:
    out[k] = d[k]
json.dump(out, open("subset20_en_full.json", "w"), indent=1)

out["oos_test"] = random.sample(out["oos_test"], N_OOS_TEST)
json.dump(out, open("subset20_en.json", "w"), indent=1)
print({k: len(v) for k, v in out.items()})

# mỗi câu có id duy nhất: <tập>_<số thứ tự>
items = []
for split in ["train", "val", "test", "oos_train", "oos_val", "oos_test"]:
    for i, (text, label) in enumerate(out[split]):
        items.append({"id": f"{split}_{i}", "split": split, "en": text, "label": label})
json.dump(items, open("items_en.json", "w"), indent=1)
print("tổng", len(items), "câu")

## 4. Dịch bằng Gemini

- Dịch theo lô, mỗi lô lưu thành file trong `batches_vi/`. Chạy lại cell này thì các lô đã xong sẽ được bỏ qua.
- Nhãn intent **không** được gửi cho mô hình, chỉ gửi câu.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
os.makedirs("batches_vi", exist_ok=True)

PROMPT = """Dịch các câu tiếng Anh sau sang tiếng Việt, theo cách người Việt thật sự nhắn tin cho trợ lý ngân hàng/thẻ tín dụng.
Quy tắc:
- Chữ thường, giữ dấu tiếng Việt, không dấu câu ở cuối câu.
- Dịch theo ý, đừng dịch từng chữ; đa dạng cách diễn đạt tự nhiên.
- Tiền tệ đổi sang đồng; tên thương hiệu Mỹ đổi thành ngân hàng/thẻ phổ biến ở Việt Nam hoặc nói chung là "thẻ".
- Giữ nguyên số và ngày.
- Đầu ra: mỗi dòng đúng dạng `id | câu tiếng Việt`, giữ nguyên id và thứ tự, không giải thích, không bỏ dòng nào.

"""

LINE_RE = re.compile(r"^\s*([a-z_]+_\d+)\s*\|\s*(.+?)\s*$")

def call_gemini(chunk):
    src = "\n".join(f"{it['id']} | {it['en']}" for it in chunk)
    for attempt in range(6):
        try:
            r = client.models.generate_content(
                model=MODEL,
                contents=PROMPT + src,
                config=types.GenerateContentConfig(temperature=0.3),
            )
            return r.text.strip()
        except Exception as e:
            print("  lỗi, thử lại:", str(e)[:120])
            time.sleep(20 * (attempt + 1))
    raise RuntimeError("Gemini lỗi liên tục, dừng lại")

scope = items[:N_TEST_SENT] if TEST_MODE else items
print("sẽ dịch", len(scope), "câu")

for b in range(0, len(scope), BATCH_SIZE):
    fn = f"batches_vi/batch_{b // BATCH_SIZE + 1:03d}.txt"
    if os.path.exists(fn):
        continue
    text = call_gemini(scope[b:b + BATCH_SIZE])
    open(fn, "w", encoding="utf-8").write(text)
    print("xong", fn)
    time.sleep(5)

## 5. Gộp bản dịch và kiểm tra thiếu sót

In [ ]:
def load_translations():
    vi = {}
    for fn in sorted(glob.glob("batches_vi/*.txt")):
        for line in open(fn, encoding="utf-8"):
            m = LINE_RE.match(line)
            if m:
                vi[m.group(1)] = m.group(2)
    return vi

vi = load_translations()
missing = [it["id"] for it in scope if it["id"] not in vi]
print("đã dịch:", len([it for it in scope if it["id"] in vi]), "/", len(scope))
print("thiếu:", len(missing), missing[:10])

Nếu còn câu thiếu (mô hình bỏ sót dòng), chạy cell dưới để dịch bù riêng các câu đó.

In [ ]:
if missing:
    by_id = {it["id"]: it for it in items}
    todo = [by_id[i] for i in missing]
    for b in range(0, len(todo), BATCH_SIZE):
        fn = f"batches_vi/fix_{int(time.time())}_{b // BATCH_SIZE + 1:03d}.txt"
        text = call_gemini(todo[b:b + BATCH_SIZE])
        open(fn, "w", encoding="utf-8").write(text)
        print("đã dịch bù", fn)
        time.sleep(5)
    vi = load_translations()
    missing = [it["id"] for it in scope if it["id"] not in vi]
    print("còn thiếu:", len(missing), missing[:10])
else:
    print("không có câu thiếu")

## 6. Khử trùng lặp và lưu bộ dữ liệu cuối

Hai câu tiếng Anh khác nhau có thể dịch thành cùng một câu tiếng Việt. Nếu câu đó nằm ở cả train và test thì mô hình đã "thấy đáp án", kết quả sẽ đẹp giả.
Quy tắc: ưu tiên giữ ở tập đứng trước (train > val > test > oos_*), câu trùng ở tập sau bị loại.

In [ ]:
priority = ["train", "val", "test", "oos_train", "oos_val", "oos_test"]
seen, dropped = {}, []
final = {k: [] for k in priority}

for split in priority:
    for it in scope:
        if it["split"] != split or it["id"] not in vi:
            continue
        t = vi[it["id"]]
        if t in seen:
            dropped.append((t, seen[t], split, it["label"]))
            continue
        seen[t] = split
        final[split].append([t, it["label"]])

print("số câu bị loại vì trùng:", len(dropped))
for x in dropped[:10]:
    print("  ", x)

json.dump(final, open("subset20_vi.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print({k: len(v) for k, v in final.items()})

## 7. Đọc kiểm tra ngẫu nhiên (nên đọc 10-15% số câu, đặc biệt các cặp intent dễ nhầm)

In [ ]:
by_id = {it["id"]: it for it in items}
ids = [i for i in vi if i in by_id]
random.seed(1)
for i in random.sample(ids, min(25, len(ids))):
    it = by_id[i]
    print(f"[{it['label']}]")
    print("  EN:", it["en"])
    print("  VI:", vi[i])

## 8. Việc tiếp theo

- Khi chất lượng ổn: đặt `TEST_MODE = False` ở cell cấu hình, chạy lại từ cell dịch (các lô đã xong được bỏ qua).
- Tải `subset20_vi.json` về từ panel Tệp.
- Ghi lại cho báo cáo: model dịch, prompt, tỉ lệ câu đã đọc kiểm tra, số câu sửa tay, số câu bị loại vì trùng.
- Tạo thêm tập test riêng: câu rút gọn (thiếu thông tin), câu đa ý (ghép 2 câu khác intent), near-OOS.